# EMBL biotransformation explorer

Pick a metagenome dataset from **MGnify** (API v2, via `mgnipy`), supply a curated
biotransformation reference file (per-microbe or per-gene evidence), and see which
biotransformations are predicted for that sample -- with the full evidence trail.

Evidence is two-tier:
- **Tier 1 (strongest):** the specific enzyme/gene was directly detected in the sample's functional annotation.
- **Tier 2 (weaker):** a microbe known to carry out the reaction is present, but its enzyme wasn't independently confirmed.


## 1. Setup


In [1]:
# Run once per environment.
# %pip install -q -r requirements.txt


In [ ]:
import sys, json
sys.path.insert(0, '.')  # the package folder sits next to this notebook

import ipywidgets as widgets
from IPython.display import display, clear_output

from embl_biotransform import (
    EvidenceGraph, BiotransformationPredictor, load_reference_associations,
    plot_evidence_graph, plot_evidence_scores, MGnifyClient,
)
from embl_biotransform.mgnipy_client import MGnipySource


## 2. Choose a dataset

Search MGnify studies by keyword, then pick a study and an analysis (a sample that's
already been through MGnify's taxonomy + functional-annotation pipeline).
If `mgnipy` / network access isn't available in this environment, switch to
**'Load from files'** below and point at previously-downloaded JSON instead --
everything downstream is identical either way.


In [ ]:
source_mode = widgets.ToggleButtons(
    options=['MGnify (live, via mgnipy)', 'Load from files'],
    description='Data source:',
    style={'description_width': 'initial'},
)
display(source_mode)


In [ ]:
# -- live MGnify search widgets (only used if source_mode == 'MGnify (live, via mgnipy)') --
keyword_box = widgets.Text(description='Study keyword:', placeholder='e.g. soil, gut, wastewater')
search_button = widgets.Button(description='Search studies')
study_dropdown = widgets.Dropdown(description='Study:', options=[])
analysis_dropdown = widgets.Dropdown(description='Analysis:', options=[])
load_analyses_button = widgets.Button(description='List analyses')
mgnipy_output = widgets.Output()

_mgnipy_source = None

def _on_search_clicked(_):
    global _mgnipy_source
    with mgnipy_output:
        clear_output()
        try:
            if _mgnipy_source is None:
                _mgnipy_source = MGnipySource()
            results = _mgnipy_source.search_studies(keyword_box.value)
            study_dropdown.options = [
                (f"{r.get('accession', '?')} - {r.get('study_name', r.get('name', ''))}", r.get('accession'))
                for r in results
            ]
            print(f'Found {len(results)} studies.')
        except Exception as exc:
            print('Live search failed:', exc)
            print("Switch 'Data source' to 'Load from files' and provide a study/analysis JSON instead.")

def _on_load_analyses_clicked(_):
    with mgnipy_output:
        clear_output()
        try:
            results = _mgnipy_source.list_analyses(study_dropdown.value)
            analysis_dropdown.options = [(r.get('accession', '?'), r.get('accession')) for r in results]
            print(f'Found {len(results)} analyses.')
        except Exception as exc:
            print('Listing analyses failed:', exc)

search_button.on_click(_on_search_clicked)
load_analyses_button.on_click(_on_load_analyses_clicked)

display(widgets.HBox([keyword_box, search_button]))
display(study_dropdown)
display(widgets.HBox([load_analyses_button]))
display(analysis_dropdown)
display(mgnipy_output)


In [ ]:
# -- offline file-based alternative --
taxonomy_file_box = widgets.Text(description='Taxonomy JSON:', value='example_data/sample_taxonomy.json',
                                  style={'description_width': 'initial'}, layout=widgets.Layout(width='500px'))
functions_file_box = widgets.Text(description='Functions JSON:', value='example_data/sample_functions.json',
                                   style={'description_width': 'initial'}, layout=widgets.Layout(width='500px'))
display(taxonomy_file_box, functions_file_box)


## 3. Reference biotransformation data (per microbe / per gene)


In [ ]:
reference_file_box = widgets.Text(
    description='Reference file:', value='example_data/reference_associations.json',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='500px'),
)
display(reference_file_box)


## 4. Run integration + prediction


In [ ]:
run_button = widgets.Button(description='Run', button_style='success')
run_output = widgets.Output()
display(run_button, run_output)

sample_node = None
graph = None
predictions = None

def _run(_):
    global sample_node, graph, predictions
    with run_output:
        clear_output()
        graph = EvidenceGraph()

        if source_mode.value.startswith('MGnify'):
            sample_id = analysis_dropdown.value
            if sample_id is None:
                print('Pick an analysis above first (or switch to "Load from files").')
                return
            sample_node = graph.add_sample(sample_id)
            taxa = _mgnipy_source.fetch_taxonomy(sample_id)
            functions = _mgnipy_source.fetch_functional_annotations(sample_id)
        else:
            sample_id = 'offline-sample'
            sample_node = graph.add_sample(sample_id)
            with open(taxonomy_file_box.value) as f:
                taxa = json.load(f)
            with open(functions_file_box.value) as f:
                functions = json.load(f)

        graph.add_detected_taxa(sample_node, taxa)
        graph.add_detected_functions(sample_node, functions)

        references = load_reference_associations(reference_file_box.value)
        graph.add_reference_associations(references)
        graph.link_detections_to_references()

        predictor = BiotransformationPredictor(graph)
        predictions = predictor.predict(sample_node)

        print(graph.summary())
        print()
        for p in predictions:
            print(f'{p.reaction_class:35s} score={p.score:.2f}  {p.confidence_label}')
            for line in p.evidence_summary:
                print('   ', line)

run_button.on_click(_run)


## 5. Evidence graph and ranked predictions


In [ ]:
if graph is not None and sample_node is not None:
    fig = plot_evidence_graph(graph, sample_node=sample_node)


In [ ]:
if predictions:
    fig = plot_evidence_scores(predictions)
